## __Adjust zones' geometrical centroid for logical centroid (based on densidad poblacional)__
When importing zones_agebs.shp (2,203) in visum, it set their centroid as their geometrical centroid
We have to adjust x & y coordinates for that centroid in order for it to be the centroid based on the pop_density of the zone

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os

In [2]:
folder = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
red_shapefiles = os.path.join(folder,"red_shapefiles")

#### 1. Read Zones.gpkg with adjusted centroid

In [10]:
zones = gpd.read_file(os.path.join(red_shapefiles,"results","generate_connectors", "agebs_centroids.gpkg"))
zones_standarized = gpd.read_file(os.path.join(red_shapefiles,"zonas_agebs_standarized.shp"))

# Merge on clave_ageb to get the visum id (id_mun_age) for each zone
zones = zones.merge(
    zones_standarized[['clave_ageb', 'id_mun_age']],
    on='clave_ageb', 
    how='left'
)

#### 2. Convert density centroid to geomtry & reproject CRS

In [14]:
# Convertir los centroide de densidad de str a geometría
density_points = gpd.GeoSeries.from_wkt(
    zones["density_centroid"],
    crs=zones.crs
)

# Reproyectar a EPSG:4326 (crs de Visum)
density_points_4326 = density_points.to_crs("EPSG:4326")

# Agregar la geometría reproyectada al GeoDataFrame
zones["density_centroid_x"] = density_points_4326.x
zones["density_centroid_y"] = density_points_4326.y

#### 3. Adjust centroid in Visum

In [17]:
import win32com.client as com

#Red base GDL (con 2,203 zonas)
red_base = os.path.join(folder, "Red Base GDL", "RedBase 300726 - copia.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [32]:
visum_zones = Visum.Net.Zones

# Ordenar por id_mun_age
zones = zones.sort_values(by='id_mun_age')

# Ajustar el índice para que comience en 1
zones = zones.reset_index(drop=True)
zones.index = zones.index + 1

# Crear listas de tuplas (id_mun_age, density_centroid_x) y (id_mun_age, density_centroid_y)
density_centroids_x = list(zip(
    zones.index,
    zones['density_centroid_x']
))
density_centroids_y = list(zip(
    zones.index,
    zones['density_centroid_y']
))

# Insert in Visum as new columns
visum_zones.SetMultiAttValues("DENSITY_CENTROID_X", density_centroids_x)
visum_zones.SetMultiAttValues("DENSITY_CENTROID_Y", density_centroids_y)